<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I use **supervised ranking from a binary future-decline proxy**.

The target is binary:

- `1` = average daily impressions in the outcome window are more than **20% lower** than in the feature window.
- `0` = otherwise.

However, the business goal is not simply to classify pages as declining or not declining. The goal is to **rank pages by their risk of future decline**, so that the highest-risk pages can be reviewed first. For this reason, I use the models' predicted probabilities as ranking scores and evaluate them using **Precision@100**.

I compare three approaches:

- **Week-4 rule baseline:** the existing transparent benchmark. A learned model should only be preferred if it provides a meaningful improvement over this rule on the same held-out data.
- **Logistic Regression:** a simple and interpretable linear benchmark that tests whether the engineered features are useful without relying on complex interactions.
- **Random Forest:** a nonlinear model that can capture interactions between signals such as traffic level, recent momentum, CTR, search position, activity, trend shape, and client-relative context.

This method fits my lane because content decline is unlikely to depend on a single signal. The combination of multiple pre-outcome signals may provide a better ranking of pages at risk.

I keep the model settings fixed before final validation and do not perform a large hyperparameter search. The aim is to test whether added model complexity is justified by a clear improvement over the Week-4 baseline, rather than using complexity for its own sake.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a **time-aware feature/outcome boundary** together with **client-disjoint grouped validation**.

### Time-aware design

For March 2026:

- **Feature window:** March 1–15
- **Outcome window:** March 16–31

All model features are created using only information available during the first half of the month.  
The second half is used only to define the future-decline target.

This makes the experiment time-aware because the model never uses information from the period it is trying to predict.

### Grouped by client

My primary validation uses **5 client-disjoint folds**. Complete clients are assigned to either training or validation within each fold, so pages from the same client never appear in both.

The folds contain approximately:

**7 / 7 / 7 / 7 / 6 clients**

I use client grouping because pages belonging to the same client can share traffic scale, search behavior, and other patterns. A random row-level split could therefore make validation results look better than they really are by allowing the model to learn client-specific patterns from the training data and see similar pages in validation.

The client-disjoint split gives a more honest test of the main question:

> Can the model rank future-decline risk for pages belonging to clients that were not used to train that fold?

### Additional temporal check

I also use **April 2026 as a secondary temporal holdout**. The model is trained on March and then evaluated on the following month without refitting.

I treat this only as a robustness check because some clients may appear in both March and April. Therefore, the **client-disjoint March validation remains the primary evidence**, while April tests whether the learned signal also holds in a later time period.

### Reproducibility note

Before finalizing the model results, I made the monthly model-frame row order deterministic by sorting the aggregated data by `client_id` and `content_id` after loading it from DuckDB.

This changes only the row order. It does not change the observations, feature definitions, target definition, model settings, or validation design.

The results reported below come from this deterministic rerun.

## 3. Train + compare vs my baseline

Same data, same metric, same split as your Week-4 baseline. Show the table.

I compare all approaches under the **same evaluation conditions**:

- the same March dataset;
- the same client-disjoint 5-fold split;
- the same future-decline target;
- the same held-out pages in each fold;
- and the same primary metric: **Precision@100**.

For the Week-4 baseline, the CTR thresholds are fitted using the training clients only, then applied to the held-out clients. Logistic Regression and Random Forest are trained on exactly the same training clients and evaluated on exactly the same validation clients.

### Primary grouped-validation results

| Approach | Mean Precision@100 | Std. P@100 | Min P@100 | Max P@100 | Average Precision | ROC-AUC |
|---|---:|---:|---:|---:|---:|---:|
| **Random Forest — Full Signal** | **74.6%** | **13.6%** | 56.0% | 89.0% | **0.527** | **0.704** |
| Logistic Regression | 69.6% | 22.1% | 40.0% | 93.0% | 0.504 | 0.685 |
| Random Forest — Momentum Only | 66.2% | 12.8% | 52.0% | 84.0% | 0.451 | 0.622 |
| Week-4 Rule Baseline | 35.2% | 6.4% | 29.0% | 46.0% | 0.356 | 0.543 |

The average decline base rate across the validation folds is **32.2%**.

### Fold-by-fold Precision@100

| Fold | Week-4 Baseline | Logistic Regression | RF Momentum Only | RF Full Signal |
|---|---:|---:|---:|---:|
| 1 | 29.0% | 40.0% | 52.0% | **56.0%** |
| 2 | 33.0% | 64.0% | 65.0% | **75.0%** |
| 3 | 35.0% | 61.0% | 57.0% | **67.0%** |
| 4 | 33.0% | **90.0%** | 84.0% | 86.0% |
| 5 | 46.0% | **93.0%** | 73.0% | **89.0%** |

### Comparison with my Week-4 baseline

Logistic Regression improves mean Precision@100 by **+34.4 percentage points** over the Week-4 baseline and beats it in all 5 folds.

Random Forest with the full feature set improves mean Precision@100 by **+39.4 percentage points** and also beats the Week-4 baseline in all 5 folds.

The full-signal Random Forest outperforms the momentum-only Random Forest by an average of **+8.4 percentage points**, winning in all 5 folds. This provides directional evidence that signals beyond recent momentum add useful ranking information.

Based on the primary metric, **Random Forest — Full Signal is the strongest model**, reaching **74.6% mean Precision@100 compared with 35.2% for my Week-4 baseline**.

This is about a **2.1× Precision@100 ratio** under the same data, split, and evaluation metric. The result should be interpreted as measured performance within this validation design, not as a universal guarantee.

##4. Errors and interpretation
Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.



I inspected both **where the Random Forest is wrong** and **which features it relies on most**.

### Where is the model wrong?

**High-ranked false positives**

Some pages received extremely high risk scores but did not cross the future-decline threshold.

The strongest false-positive examples showed signals such as:

- impression momentum of approximately **-39% to -83%**;
- worsening search position;
- one or more declining impression steps.

These pages looked very similar to genuine decline cases during the feature window, so the model ranked them highly. However, the decline did not persist into the outcome window.

This suggests the model can mistake **short-term volatility or temporary deterioration** for sustained decline.

**Missed declines**

The opposite pattern appears in some false negatives. Several pages that later declined showed apparently healthy first-half signals:

- impression momentum from approximately **+15% to +112%**;
- no declining impression steps;
- improving search position.

The model therefore assigned them very low risk.

These are difficult cases because the decline was **not clearly visible in the available pre-outcome signals**. A page can look healthy in the first half of the month and still decline later.

### What does the model lean on?

The Random Forest relies most strongly on recent impression behavior and relative context.

Its top features are:

| Feature | Mean importance |
|---|---:|
| `imp_recent_vs_prior_log` | 0.141 |
| `imp_step23_log` | 0.137 |
| `recent_vs_prior_client_percentile` | 0.117 |
| `log_imp_first_half` | 0.088 |
| `avg_position_first_half` | 0.064 |
| `ctr_first_half` | 0.054 |

The strongest signal is therefore **recent impression movement**, followed by how that movement compares with other pages from the same client.

My interpretation is that the model mainly asks:

> Is this page losing traffic recently, and does that deterioration look unusual relative to its own client context?

The error analysis also shows the limitation of this approach: **strong recent decline does not always continue, and future decline can sometimes begin without a clear warning in the first-half data.**

I treat feature importance as a description of **model behavior**, not evidence that these features cause future decline.

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.